In [11]:
### 1.Работа с API. Извлечение данных изи OpenAPI T-Invest.
import os
import requests
import re
import logging
import pandas as pd
import numpy as np
from sqlalchemy import create_engine, text
from datetime import datetime, timedelta, timezone
from dotenv import load_dotenv
logging.basicConfig(level=logging.INFO, format='%(asctime)s - %(levelname)s - %(message)s')

load_dotenv()

TOKEN = os.getenv('T_TOKEN')
POSTGRES_USER = os.getenv('POSTGRES_USER')
POSTGRES_PASSWORD = os.getenv('POSTGRES_PASSWORD')
POSTGRES_DB = os.getenv('POSTGRES_DB')
URL = 'https://invest-public-api.tbank.ru/rest/tinkoff.public.invest.api.contract.v1.InstrumentsService/GetAssets'

def camel_to_snake(name):
    """Преобразует camelCase в snake_case"""
    # Вставляем _ перед заглавными буквами (кроме первой)
    name = re.sub('(.)([A-Z][a-z]+)', r'\1_\2', name)
    name = re.sub('([a-z0-9])([A-Z])', r'\1_\2', name)

    return name.lower()

headers = {
        "Authorization": f"Bearer {TOKEN}",
        "Content-Type": "application/json"
    }

data = {
  "instrumentType": "INSTRUMENT_TYPE_SHARE",
  "instrumentStatus": "INSTRUMENT_STATUS_ALL"
}

response = requests.post(url = URL, headers = headers, json = data, verify = False)

if response.status_code == 200:
    # return response.json()
    df = pd.json_normalize(response.json()['assets'], sep='_', record_path='instruments', meta=['uid', 'name'], meta_prefix='asset_')
    # Переименовываем колонки для удобства
    # df = df.rename(columns={
    #     'uid': 'asset_uid',
    #     'name': 'asset_name'
    # })
else:
    print(f"Ошибка: {response.status_code}")
    print(response.text)
    # return None
    print(None)

# Применяем ко всем столбцам
df.columns = [camel_to_snake(col) for col in df.columns]

/home/kam/datalearnhome/venv/lib/python3.12/site-packages/urllib3/connectionpool.py:1110: InsecureRequestWarning: Unverified HTTPS request is being made to host 'invest-public-api.tbank.ru'. Adding certificate verification is strongly advised. See: https://urllib3.readthedocs.io/en/latest/advanced-usage.html#tls-warnings
  warnings.warn(


In [13]:
df[df['figi'] == 'BBG004730N88']

,uid,figi,instrument_type,ticker,class_code,links,instrument_kind,position_uid,asset_uid,asset_name
584,e6123145-9665-43e0-8413-cd61b8aa9b13,BBG004730N88,share,SBER,TQBR,[],INSTRUMENT_TYPE_SHARE,41eb2102-5333-4713-bf15-72b204c4bf7b,40d89385-a03a-4659-bf4e-d3ecba011782,Акции обыкновенные ПАО Сбербанк


In [6]:
df['instruments']

0       [{'uid': '44a1e3e8-b813-46dd-b1f8-d0e9d91c06d2...
1       [{'uid': '9b3c9abf-f865-4e62-a49d-8f41626a2160...
2       [{'uid': 'e2955340-5e7a-47e7-82b1-fead7e9cb232...
3       [{'uid': '2dfbc1fd-b92a-436e-b011-928c79e805f2...
4       [{'uid': 'd2ded18a-3399-4b80-9c20-cc8bf4fae242...
                              ...                        
2806    [{'uid': 'b91ab0ff-78aa-43c1-9531-ad8ac80ccf5e...
2807    [{'uid': '772bd8ee-2839-4a73-8ff1-34bd4af80a04...
2808    [{'uid': 'e032bcf5-2ab8-43b3-b7c7-559177a5a5ed...
2809    [{'uid': '62025a89-f9fc-4f06-a3b6-cf668fe1e3eb...
2810    [{'uid': '8901a85d-b426-485a-aa8d-3d5260ec9d68...
Name: instruments, Length: 2811, dtype: object